# K-fold · seed 과적합 확인 노트북

`scripts/run_kfold.py`가 만든 `outputs/kfold/{exp_name}/` 결과를 읽어서 과적합 여부를 판단해요.

**과적합을 보는 세 가지 눈**
1. **train mAP − val mAP 차이(gap)**: 외운 사진(train)은 잘 맞히는데 처음 보는 사진(val)은 못 맞히면 차이가 커져요. 클수록 과적합이에요.
2. **val loss 곡선 모양**: train loss는 계속 내려가는데 val loss가 바닥을 찍고 다시 올라가면, 그 지점부터 과적합이 시작된 거예요.
3. **fold 간 편차**: fold마다 val 점수가 크게 다르면, 점수가 "어떤 조합이 val에 들어갔냐"에 따라 흔들린다는 뜻이에요. 한 번 나눠서 얻은 점수를 그대로 믿기 어려워요.

**seed 간 편차**는 같은 데이터로 랜덤성만 바꿨을 때의 흔들림이에요. 두 모델의 점수 차이가 seed 편차보다 작으면 "차이가 없다"고 보는 게 안전해요.

**순서**: 1. 설정 → 2. 요약표 → 3. train vs val 막대 → 4. 편차 → 5. loss 곡선 → 6. val mAP 곡선 → 7. 학습률 → 8. 자동 판정

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# run_kfold.py 결과 폴더 (exp_name만 바꾸면 돼요)
EXP_DIR = Path('outputs/kfold/1027_sun_yolo11s_v0_img640')

# 판정 기준 (데이터 상황에 맞게 조절)
GAP_WARN = 0.05     # train-val mAP50-95 차이가 이보다 크면 '약한 과적합'
GAP_BAD = 0.10      # 이보다 크면 '과적합'
STD_WARN = 0.03     # fold 간 val mAP50-95 표준편차가 이보다 크면 '분할에 따라 불안정'
RISE_WARN = 0.10    # val loss가 최저점보다 10% 넘게 올라가 있으면 '과적합 신호'

COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#9b59b6', '#e0a800', '#6c757d']
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6,
    'axes.unicode_minus': False,   # 한글 폰트에서 마이너스 기호 깨짐 방지
})
# 그래프 한글 폰트 (Mac: AppleGothic, Windows: Malgun Gothic, Colab: 나눔 폰트 설치 필요)
import platform
from matplotlib import font_manager
KFONT = {'Darwin': 'AppleGothic', 'Windows': 'Malgun Gothic'}.get(platform.system(), 'NanumGothic')
if KFONT in {f.name for f in font_manager.fontManager.ttflist}:
    plt.rcParams['font.family'] = KFONT

## 1. 결과 불러오기

`kfold_summary.csv`는 run마다 한 줄이에요. 각 run의 epoch별 기록(`results.csv`)도 같이 읽어요.
YOLOv8/YOLO11의 세 번째 손실은 `dfl_loss`, YOLO26은 `l1_loss`라서 `third_loss`로 이름을 통일해요.

In [ ]:
summary = pd.read_csv(EXP_DIR / 'kfold_summary.csv').sort_values(['fold', 'seed']).reset_index(drop=True)
summary['run'] = 'fold' + summary['fold'].astype(str) + '_seed' + summary['seed'].astype(str)

def load_curve(row):
    path = Path(row['results_csv']) if isinstance(row['results_csv'], str) else None
    if path is None or not path.exists():   # 다른 컴퓨터로 옮겨온 경우 상대 경로로 다시 찾기
        hits = list((EXP_DIR / row['run']).rglob('results.csv'))
        path = hits[0] if hits else None
    if path is None:
        return None
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    third = 'dfl' if 'train/dfl_loss' in df.columns else 'l1'
    return df.rename(columns={
        f'train/{third}_loss': 'train/third_loss',
        f'val/{third}_loss': 'val/third_loss',
        'metrics/mAP50-95(B)': 'mAP50_95',
        'metrics/mAP50(B)': 'mAP50',
    })

curves = {r['run']: c for _, r in summary.iterrows() if (c := load_curve(r)) is not None}
print(f'run {len(summary)}개, fold {summary.fold.nunique()}개, seed {summary.seed.nunique()}개, 곡선 {len(curves)}개 읽음')

## 2. 요약표

- `train_mAP50_95`: best.pt로 train 사진을 다시 채점한 점수
- `val_mAP50_95`: 처음 보는 조합(val)으로 채점한 점수. **모델 비교의 핵심 지표**
- `gap_mAP50_95`: 둘의 차이. 클수록 과적합
- `best_epoch` / `epochs`: val mAP가 가장 높았던 epoch과 실제 학습한 epoch 수. 조기 종료(patience)가 걸렸으면 epochs가 설정값보다 작아요

In [ ]:
def curve_info(run):
    c = curves.get(run)
    if c is None:
        return pd.Series({'best_epoch': np.nan, 'epochs': np.nan})
    return pd.Series({'best_epoch': int(c.loc[c['mAP50_95'].idxmax(), 'epoch']), 'epochs': int(c['epoch'].max())})

summary = summary.join(summary['run'].apply(curve_info))
cols = ['run', 'n_train', 'n_val', 'train_mAP50_95', 'val_mAP50_95', 'gap_mAP50_95',
        'val_mAP50', 'val_precision', 'val_recall', 'best_epoch', 'epochs']
display(summary[cols].style.format(precision=4)
        .background_gradient(subset=['gap_mAP50_95'], cmap='Reds')
        .background_gradient(subset=['val_mAP50_95'], cmap='Greens'))

## 3. fold별 train vs val 점수

막대 두 개 사이 간격이 **과적합 정도**예요. seed가 여러 개면 평균을 막대로, seed 간 범위를 오차 막대로 그려요.

In [ ]:
by_fold = summary.groupby('fold')[['train_mAP50_95', 'val_mAP50_95', 'gap_mAP50_95']]
mean, lo, hi = by_fold.mean(), by_fold.min(), by_fold.max()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x, w = np.arange(len(mean)), 0.38
for i, (col, label) in enumerate([('train_mAP50_95', 'train'), ('val_mAP50_95', 'val')]):
    err = [mean[col] - lo[col], hi[col] - mean[col]]
    axes[0].bar(x + (i - 0.5) * w, mean[col], w, yerr=err, capsize=3, label=label, color=COLORS[i])
axes[0].set_xticks(x, [f'fold{f}' for f in mean.index])
axes[0].set_ylabel('mAP50-95')
axes[0].set_title('train vs val mAP50-95')
axes[0].set_ylim(max(0, mean[['train_mAP50_95', 'val_mAP50_95']].min().min() - 0.1), 1.0)
axes[0].legend()

gap = mean['gap_mAP50_95']
bar_colors = ['#c0392b' if g > GAP_BAD else '#e0a800' if g > GAP_WARN else '#1baf7a' for g in gap]
axes[1].bar(x, gap, color=bar_colors)
axes[1].axhline(GAP_WARN, ls='--', c='#e0a800', lw=1, label=f'주의 {GAP_WARN}')
axes[1].axhline(GAP_BAD, ls='--', c='#c0392b', lw=1, label=f'과적합 {GAP_BAD}')
axes[1].set_xticks(x, [f'fold{f}' for f in gap.index])
axes[1].set_title('train - val 차이 (gap)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. 점수가 얼마나 흔들리나 (fold 편차 vs seed 편차)

- **fold 간 표준편차**: 같은 seed 안에서 fold끼리 비교 → 데이터 나누기의 영향
- **seed 간 표준편차**: 같은 fold 안에서 seed끼리 비교 → 랜덤성의 영향

최종 보고에는 `val mAP50-95 = 평균 ± fold 표준편차` 형태로 쓰면 돼요.

In [ ]:
stats = {
    'val mAP50-95 평균': summary['val_mAP50_95'].mean(),
    'train mAP50-95 평균': summary['train_mAP50_95'].mean(),
    'gap 평균': summary['gap_mAP50_95'].mean(),
    'fold 간 표준편차 (val)': summary.groupby('seed')['val_mAP50_95'].std().mean(),
}
if summary['seed'].nunique() > 1:
    stats['seed 간 표준편차 (val)'] = summary.groupby('fold')['val_mAP50_95'].std().mean()
display(pd.Series(stats, name='값').to_frame().style.format('{:.4f}'))

if summary['seed'].nunique() > 1:
    piv = summary.pivot(index='fold', columns='seed', values='val_mAP50_95')
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for i, f in enumerate(piv.index):
        ax.plot(piv.columns.astype(str), piv.loc[f], 'o-', c=COLORS[i % len(COLORS)], label=f'fold{f}')
    ax.set_xlabel('seed')
    ax.set_ylabel('val mAP50-95')
    ax.set_title('seed를 바꿨을 때 val 점수 (선이 평평할수록 안정)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. train / val loss 곡선 (epoch별 선그래프)

실선이 train loss, 점선이 val loss예요. 색은 fold, 같은 fold의 seed는 같은 색이에요.
빨간 점은 **val loss가 가장 낮았던 epoch**이에요. 그 뒤로 val loss가 올라가면 과적합 신호예요.

> train loss는 증강(mosaic, 회전 등)을 건 사진으로, val loss는 원본 사진으로 계산해서 **절대값을 직접 비교하면 안 돼요.**
> 두 선의 **모양**(같이 내려가는지, val만 다시 올라가는지)을 보세요.

In [ ]:
LOSSES = [('box_loss', 'box loss (박스 위치)'), ('cls_loss', 'cls loss (어떤 약인지)'), ('third_loss', 'dfl/l1 loss (박스 경계)')]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (loss, title) in zip(axes, LOSSES):
    for run, c in curves.items():
        fold = int(run.split('_')[0].removeprefix('fold'))
        color = COLORS[fold % len(COLORS)]
        first_seed = summary.query('fold == @fold')['run'].iloc[0] == run
        ax.plot(c['epoch'], c[f'train/{loss}'], '-', c=color, lw=1.2, alpha=0.8,
                label=f'fold{fold} train' if first_seed else None)
        ax.plot(c['epoch'], c[f'val/{loss}'], '--', c=color, lw=1.2, alpha=0.8,
                label=f'fold{fold} val' if first_seed else None)
        i = c[f'val/{loss}'].idxmin()
        ax.plot(c.loc[i, 'epoch'], c.loc[i, f'val/{loss}'], 'o', c='#c0392b', ms=4)
    ax.set_title(title)
    ax.set_xlabel('epoch')
    # 첫 epoch의 val cls loss가 수십까지 튀는 경우가 있어서, 위쪽을 2~98% 범위로 잘라서 봐요
    vals = pd.concat([c[[f'train/{loss}', f'val/{loss}']].stack() for c in curves.values()])
    ax.set_ylim(vals.quantile(0.0) * 0.9, vals.quantile(0.98) * 1.1)
axes[0].set_ylabel('loss')
axes[-1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 6. val mAP 곡선

fold마다 언제 점수가 최고였는지, 끝까지 오르고 있는지(= epoch를 더 늘려도 됨) 확인해요.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for run, c in curves.items():
    fold = int(run.split('_')[0].removeprefix('fold'))
    ax.plot(c['epoch'], c['mAP50_95'], c=COLORS[fold % len(COLORS)], lw=1.2, alpha=0.8, label=run)
ax.set_xlabel('epoch')
ax.set_ylabel('val mAP50-95')
ax.set_title('epoch별 val mAP50-95')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 7. 학습률(learning rate) 곡선

`lr/pg0`, `lr/pg1`, `lr/pg2`는 파라미터 그룹별 학습률이에요 (pg0: 가중치, pg1: BatchNorm, pg2: bias).
처음 몇 epoch은 warmup으로 올라가고, 그 뒤로 서서히 내려가는 모양이 정상이에요.
설정이 같으면 모든 run이 같은 모양이라 첫 run만 그려요.

In [ ]:
c = next(iter(curves.values()))
fig, ax = plt.subplots(figsize=(8, 3.5))
for i, pg in enumerate(['lr/pg0', 'lr/pg1', 'lr/pg2']):
    if pg in c.columns:
        ax.plot(c['epoch'], c[pg], c=COLORS[i], label=pg)
ax.set_xlabel('epoch')
ax.set_ylabel('learning rate')
ax.set_yscale('log')
ax.set_title('epoch별 학습률')
ax.legend()
plt.tight_layout()
plt.show()

## 8. 자동 판정

위 기준값(GAP_WARN, GAP_BAD, STD_WARN, RISE_WARN)으로 run마다 판정해요. 숫자는 출발점일 뿐이니 그래프와 같이 보세요.

In [ ]:
def val_loss_rise(run):
    """마지막 epoch의 val loss(세 개 합)가 최저점보다 몇 % 높은지"""
    c = curves.get(run)
    if c is None:
        return np.nan
    total = c[['val/box_loss', 'val/cls_loss', 'val/third_loss']].sum(axis=1)
    return total.iloc[-1] / total.min() - 1

summary['val_loss_rise'] = summary['run'].apply(val_loss_rise)

def verdict(r):
    if r['gap_mAP50_95'] > GAP_BAD or r['val_loss_rise'] > RISE_WARN:
        return '과적합'
    if r['gap_mAP50_95'] > GAP_WARN:
        return '약한 과적합'
    return '괜찮음'

summary['판정'] = summary.apply(verdict, axis=1)
display(summary[['run', 'train_mAP50_95', 'val_mAP50_95', 'gap_mAP50_95', 'val_loss_rise', '판정']]
        .style.format({'train_mAP50_95': '{:.4f}', 'val_mAP50_95': '{:.4f}',
                       'gap_mAP50_95': '{:+.4f}', 'val_loss_rise': '{:+.1%}'}))

fold_std = summary.groupby('seed')['val_mAP50_95'].std().mean()
n_bad = (summary['판정'] == '과적합').sum()
print(f"val mAP50-95 = {summary['val_mAP50_95'].mean():.4f} ± {fold_std:.4f} (fold 표준편차)")
print(f"평균 gap = {summary['gap_mAP50_95'].mean():+.4f},  과적합 판정 {n_bad}/{len(summary)} run")
if fold_std > STD_WARN:
    print('→ fold마다 점수 차이가 커요. 특정 조합/약이 어려운지 fold별 혼동행렬을 확인해 보세요.')
if n_bad:
    print('→ 과적합 대처: 증강 강화, epoch 줄이기 또는 patience 낮추기, 더 작은 모델(s→n), weight_decay 올리기')
elif summary['gap_mAP50_95'].mean() < GAP_WARN:
    print('→ 과적합은 크지 않아요. 아직 val mAP가 오르는 중이면 epoch를 늘리거나 더 큰 모델(m)을 시도해 볼 만해요.')